# Flood Command Center - Exploratory Data Analysis & Data Fusion

This notebook demonstrates:
1. Ingestion of multi-modal disaster telemetry (IMD, WRIS, Bhuvan, DEM).
2. Spatio-temporal data fusion and feature engineering.
3. Correlation analysis between rainfall, river discharge, elevation, and flood risk scores.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Ensure root path is accessible
sys.path.insert(0, os.path.abspath('..'))

from src.data_collectors import IMDDataCollector, WRISDataCollector, BhuvanDataCollector, DEMDataCollector
from src.preprocessing import DataFusionPipeline
from src.models import FloodPredictorModel

## 1. Fetching Telemetry Data across Collectors

In [ ]:
imd_df = IMDDataCollector().fetch(use_simulation=True)
wris_df = WRISDataCollector().fetch(use_simulation=True)
bhuvan_df = BhuvanDataCollector().fetch(use_simulation=True)
dem_df = DEMDataCollector().fetch(use_simulation=True)

print(f"IMD Shape: {imd_df.shape}")
print(f"WRIS Shape: {wris_df.shape}")
print(f"Bhuvan Shape: {bhuvan_df.shape}")
print(f"DEM Shape: {dem_df.shape}")

## 2. Spatio-Temporal Data Fusion

In [ ]:
pipeline = DataFusionPipeline()
fused_df = pipeline.process_and_fuse(imd_df, wris_df, bhuvan_df, dem_df)
fused_df.head(10)

## 3. Flood Risk Predictions

In [ ]:
predictor = FloodPredictorModel(model_dir='../models')
predictions = []
for idx, row in fused_df.iterrows():
    res = predictor.predict_district(row.to_dict())
    predictions.append(res)

pred_df = pd.DataFrame(predictions)
pred_df.head(10)